# Siapkan Dataset 9-Kelas → Google Drive

Notebook ini membangun folder `datasets/` berisi **8 kelas pisang + 1 kelas negatif `Not Banana Leaf`**, lalu menyimpannya (sebagai zip) ke Google Drive.

**Kenapa kelas negatif?** Saat sidang, sistem salah membaca daun pepaya, daun kelapa, daun lain, lantai, tangan, dan kaki sebagai *penyakit pisang*. Heuristik OOD berbasis ImageNet tidak cukup andal. Solusinya: latih kelas negatif eksplisit agar model belajar batas "ini daun pisang / bukan".

**Alur:** download dataset di Colab (cepat) → susun + sampling seimbang di `/content` → **zip** → salin satu file zip ke Drive (menyalin puluhan ribu file kecil langsung ke Drive sangat lambat).

Jalankan sel berurutan dari atas.

## 1. Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 2. Kredensial Kaggle
Jalankan sel ini lalu unggah file `kaggle.json` (dari Akun Kaggle → Settings → Create New Token).

In [ ]:
import os

# Opsi A: kaggle.json sudah tersimpan di Drive (mis. /content/drive/MyDrive/kaggle.json)
DRIVE_KAGGLE = '/content/drive/MyDrive/kaggle.json'
if os.path.exists(DRIVE_KAGGLE):
    os.makedirs('/root/.kaggle', exist_ok=True)
    !cp "{DRIVE_KAGGLE}" /root/.kaggle/kaggle.json
else:
    # Opsi B: unggah manual dari komputer
    from google.colab import files
    files.upload()  # pilih kaggle.json
    os.makedirs('/root/.kaggle', exist_ok=True)
    !cp kaggle.json /root/.kaggle/kaggle.json

!chmod 600 /root/.kaggle/kaggle.json
!pip -q install kaggle
!kaggle --version

## 3. Konfigurasi

In [ ]:
from pathlib import Path

STAGING  = Path('/content/.staging')        # area unduh sementara
DATASETS = Path('/content/datasets')        # hasil akhir (akan di-zip)
DRIVE_OUT = Path('/content/drive/MyDrive/banana-datasets')  # tujuan di Drive
SEED = 42
IMG_EXT = {'.jpg', '.jpeg', '.png', '.bmp', '.webp'}
NEG_TOTAL = None  # None = otomatis (~ rata-rata 1 kelas pisang)

# 8 kelas pisang (harus sama persis dengan artifacts/labels.json)
BANANA_LABELS = [
    'Augmented Banana Black Sigatoka Disease',
    'Augmented Banana Bract Mosaic Virus Disease',
    'Augmented Banana Cordana Disease',
    'Augmented Banana Healthy Leaf',
    'Augmented Banana Insect Pest Disease',
    'Augmented Banana Moko Disease',
    'Augmented Banana Panama Disease',
    'Augmented Banana Yellow Sigatoka Disease',
]
NEG_LABEL = 'Not Banana Leaf'

BANANA_SOURCES = {
    'main':    'sujaykapadnis/banana-disease-recognition-dataset',  # 7 kelas (tanpa Cordana)
    'cordana': 'shifatearman/bananalsd',                            # sumber Cordana
}

# Sumber NEGATIF. weight = proporsi kuota negatif dari sumber ini.
NEG_SOURCES = [
    # Daun non-pisang (HARD NEGATIVE, porsi terbesar)
    {'slug': 'ajithdari/papaya-leaf-disease-dataset',                     'weight': 0.18, 'group': 'daun-pepaya'},
    {'slug': 'shravanatirtha/coconut-leaf-dataset-for-pest-identification','weight': 0.15, 'group': 'daun-kelapa'},
    {'slug': 'nirmalsankalana/plantdoc-dataset',                          'weight': 0.22, 'group': 'daun-multispesies'},
    # Bukan daun: tangan/telapak (penguji uji tangan & kaki)
    {'slug': 'shyambhu/hands-and-palm-images-dataset',                    'weight': 0.15, 'group': 'tangan'},
    # Bukan daun: objek/orang/hewan acak
    {'slug': 'prasunroy/natural-images',                                  'weight': 0.20, 'group': 'objek-acak'},
    # Bukan daun: scene/lantai (2.5GB; aktifkan bila perlu)
    {'slug': 'itsahmad/indoor-scenes-cvpr-2019',                          'weight': 0.10, 'group': 'scene-lantai', 'enabled': False},
]

## 4. Fungsi bantu

In [ ]:
import random, shutil, subprocess

def kaggle_download(slug, dest):
    """Unduh & ekstrak dataset Kaggle ke dest. Idempoten."""
    dest = Path(dest); dest.mkdir(parents=True, exist_ok=True)
    if any(dest.iterdir()):
        print(f'  [skip] {slug} (sudah ada)'); return True
    print(f'  [download] {slug}')
    try:
        subprocess.run(['kaggle','datasets','download','-d',slug,'-p',str(dest),'--unzip'], check=True)
        return True
    except subprocess.CalledProcessError as e:
        print(f'  [GAGAL] {slug}: {e}'); return False

def list_images(root):
    root = Path(root)
    return [p for p in root.rglob('*') if p.suffix.lower() in IMG_EXT and p.is_file()]

def norm(s):
    return ''.join(c for c in s.lower() if c.isalnum())

def find_class_dir(staging, label):
    """Cari folder paling cocok dengan nama label (tahan beda penamaan antar versi)."""
    key = label.replace('Augmented Banana','').replace('Disease','').replace('Leaf','').strip()
    key_n = norm(key); cands = []
    for d in Path(staging).rglob('*'):
        if not d.is_dir(): continue
        dn = norm(d.name)
        if not dn: continue
        if dn == norm(label):                         cands.append((3, d))
        elif key_n and key_n in dn:                   cands.append((2, d))
        elif key_n and dn in key_n and len(dn) >= 4:  cands.append((1, d))
    if not cands: return None
    cands.sort(key=lambda t: (t[0], len(list_images(t[1]))), reverse=True)
    return cands[0][1]

def copy_sample(images, n, dest, prefix, rng):
    """Salin maksimal n gambar acak ke dest dengan nama berprefiks (anti-bentrok)."""
    dest = Path(dest); dest.mkdir(parents=True, exist_ok=True)
    chosen = images if len(images) <= n else rng.sample(images, n)
    for i, src in enumerate(chosen):
        try: shutil.copy2(src, dest / f'{prefix}_{i:05d}{src.suffix.lower()}')
        except OSError: pass
    return len(chosen)

## 5. Bangun 8 kelas pisang

In [ ]:
print('=== Tahap 1: pisang ===')
main_stage = STAGING / 'banana_main'
cordana_stage = STAGING / 'banana_cordana'
kaggle_download(BANANA_SOURCES['main'], main_stage)
kaggle_download(BANANA_SOURCES['cordana'], cordana_stage)

banana_counts = {}
for label in BANANA_LABELS:
    search_root = cordana_stage if 'Cordana' in label else main_stage
    src_dir = find_class_dir(search_root, label)
    dest = DATASETS / label; dest.mkdir(parents=True, exist_ok=True)
    if src_dir is None:
        print(f'  [!] sumber untuk {label!r} TIDAK ditemukan'); banana_counts[label] = 0; continue
    rng = random.Random(SEED)
    imgs = list_images(src_dir)
    copy_sample(imgs, len(imgs), dest, norm(label)[:12], rng)
    banana_counts[label] = len(list_images(dest))
    print(f'  [{label}] <- {src_dir.name} ({banana_counts[label]})')

## 6. Bangun kelas negatif `Not Banana Leaf`

In [ ]:
valid = [c for c in banana_counts.values() if c > 0]
avg = round(sum(valid)/len(valid)) if valid else 1000
neg_total = NEG_TOTAL or avg
print(f'=== Tahap 2: negatif (target ~{neg_total}) ===')

dest = DATASETS / NEG_LABEL
if dest.exists(): shutil.rmtree(dest)
dest.mkdir(parents=True, exist_ok=True)

active = [s for s in NEG_SOURCES if s.get('enabled', True)]
total_w = sum(s['weight'] for s in active)
for s in active:
    stage = STAGING / ('neg_' + s['group'])
    if not kaggle_download(s['slug'], stage):
        print(f"  [lewati] {s['group']}"); continue
    imgs = list_images(stage)
    if not imgs:
        print(f"  [lewati] {s['group']} (kosong)"); continue
    quota = max(1, round(neg_total * s['weight'] / total_w))
    rng = random.Random(SEED + hash(s['group']) % 1000)
    n = copy_sample(imgs, quota, dest, s['group'], rng)
    print(f"  [{s['group']:18s}] tersedia {len(imgs):5d} -> diambil {n} (kuota {quota})")
print('  TOTAL negatif:', len(list_images(dest)))

## 7. Ringkasan

In [ ]:
print('=== Ringkasan datasets/ ===')
grand = 0
for label in BANANA_LABELS + [NEG_LABEL]:
    n = len(list_images(DATASETS / label)); grand += n
    print(f'  {label:45s} : {n}')
print(f'  {"TOTAL":45s} : {grand}')

## 8. Zip → simpan ke Google Drive
Menyalin satu file zip jauh lebih cepat & andal daripada puluhan ribu file kecil lewat Drive FUSE.

In [ ]:
DRIVE_OUT.mkdir(parents=True, exist_ok=True)
zip_base = '/content/banana_datasets_9class'
print('Membuat zip... (bisa beberapa menit)')
shutil.make_archive(zip_base, 'zip', root_dir=str(DATASETS))
zip_path = zip_base + '.zip'
size_mb = os.path.getsize(zip_path) / 1e6
print(f'Zip dibuat: {zip_path} ({size_mb:.0f} MB)')

dst = DRIVE_OUT / 'banana_datasets_9class.zip'
!cp "{zip_path}" "{dst}"
print('Tersimpan di Drive:', dst)
print('\nDi notebook training, ekstrak dengan:')
print(f'  !cp "{dst}" /content/ && unzip -q /content/banana_datasets_9class.zip -d /content/datasets')

---
**Langkah berikutnya:** di [train-collabs.ipynb](train-collabs.ipynb), ekstrak zip dari Drive ke `/content/datasets`, lalu set `NUM_CLASSES = 9` dan latih ulang ensemble. `labels.json` akan otomatis berisi 9 kelas (8 pisang + `Not Banana Leaf`).